In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('inventory.db')

df = pd.read_sql_query("""
SELECT * FROM vendor_sales_summary
WHERE GrossProfit > 0
AND ProfitMargin > 0
AND TotalSalesQuantity > 0
""", conn)

In [2]:
df.to_csv("vendorSalesSummary.csv", index=False)

In [3]:
brand_performance = df.groupby("Description").agg({
    'TotalSalesDollars': 'sum',
    'ProfitMargin': 'mean'
}).reset_index()

brand_performance.to_csv("BrandPerformance.csv", index=False)

In [4]:
vendor_performance = df.groupby('VendorName').agg({
    'TotalPurchaseDollars': 'sum',
    'GrossProfit': 'sum',
    'TotalSalesDollars': 'sum'
}).reset_index()

vendor_performance['Purchase_Contribution%'] = (
    vendor_performance['TotalPurchaseDollars'] /
    vendor_performance['TotalPurchaseDollars'].sum()
) * 100

vendor_performance.to_csv("PurchaseContribution.csv", index=False)

In [5]:
low_turnover_vendor = df[df['StockTurnover'] < 1].groupby('VendorName').agg({
    'StockTurnover': 'mean'
}).reset_index().sort_values(by='StockTurnover')

low_turnover_vendor.to_csv("LowTurnoverVendor.csv", index=False)

In [6]:
df["UnsoldInventoryValue"] = (
    (df["TotalPurchaseQuantity"] - df["TotalSalesQuantity"]) * df["PurchasePrice"]
)

unsold_inventory = df.groupby("VendorName")["UnsoldInventoryValue"].sum().reset_index()

unsold_inventory.to_csv("UnsoldCapital.csv", index=False)